In [1]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from datasets import load_dataset
from itertools import chain
from dataclasses import dataclass, field

In [2]:
@dataclass
class CustomArguments(TrainingArguments):
    # lora_r
    lora_r: int = field(default=8)
    # num_proc
    num_proc: int = field(default=1)
    # max_seq_length
    max_seq_length: int = field(default=10)
    # eval_stragety
    eval_stragety: str = field(default="steps")
    # eval_steps
    eval_steps: int = field(default=1000)
    # seed
    seed: int = field(default=0)
    # optim
    optim: str = field(default="adamw_torch")
    # num_train_epochs
    num_train_epochs: int = field(default=1)
    # per_device_train_batch_size
    per_device_train_batch_size: int = field(default=2)
    # learning_rate
    learning_rate: float = field(default=5e-5)
    # weight_decay
    weight_decay: float = field(default=0)
    # warmup_steps
    warmup_steps: int = field(default=1000)
    # lr_schedule_steps
    lr_schedule_steps: str = field(default="linear")
    # gradient_checkpointing
    gradient_checkpointing: bool = field(default=False)
    # bf16
    bf16: bool = field(default=False)
    # gradient_accumulation_steps
    gradient_accumulation_steps: int = field(default=1)
    # logging_steps
    logging_steps: int = field(default=30)
    # save_strategy
    save_strategy: str = field(default="steps")
    # save_steps
    save_steps: int = field(default=3)
    # save_total_limit
    save_total_limit: int = field(default=2)

In [3]:
parser = transformers.HfArgumentParser(CustomArguments)
training_args, remaining = parser.parse_args_into_dataclasses(return_remaining_strings=True)

In [4]:
model_path = "/kaggle/input/notebooks/shipengalan/download-model/models/Qwen3.5-0.8B"
train_data_dir = "/kaggle/input/datasets/shipengalan/mydata/text/zh/train_data"
eval_data_dir = "/kaggle/input/datasets/shipengalan/mydata/text/zh/eval_data"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [6]:
with training_args.main_process_first(desc="load dataset"):
    train_dataset = load_dataset("text", data_dir=train_data_dir, split="train")
    eval_dataset = load_dataset("text", data_dir=eval_data_dir, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
def tokenization(example):
    return tokenizer(example["text"])

def group_texts(examples):
    max_seq_length = training_args.max_seq_length
    concat_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concat_examples["input_ids"])
    total_length = (total_length // max_seq_length) * max_seq_length
    result = {k: [t[i:i+max_seq_length] for i in range(0, total_length, max_seq_length)] for k, t in concat_examples.items()}
    result["labels"] = result["input_ids"].copy()
    return result

In [8]:
with training_args.main_process_first(desc="dataset map tokenization"):
    train_dataset = train_dataset.map(tokenization, remove_columns=["text"], num_proc=training_args.num_proc)
    train_dataset = train_dataset.map(group_texts, batched=True)

    eval_dataset = eval_dataset.map(tokenization, remove_columns=["text"], num_proc=training_args.num_proc)
    eval_dataset = eval_dataset.map(group_texts, batched=True)

Map (num_proc=1):   0%|          | 0/127 [00:00<?, ? examples/s]

Map:   0%|          | 0/127 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/23 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

In [9]:
train_dataset[3]

{'input_ids': [110699,
  96183,
  95923,
  100222,
  96826,
  95726,
  112832,
  99970,
  95849,
  3709],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [110699,
  96183,
  95923,
  100222,
  96826,
  95726,
  112832,
  99970,
  95849,
  3709]}

In [10]:
print(tokenizer.bos_token_id, tokenizer.eos_token_id)

None 248046


In [11]:
tokenizer("我爱北京天安门"), tokenizer.decode([128028])

({'input_ids': [102286, 98116, 128028], 'attention_mask': [1, 1, 1]}, '天安门')